# Work with Embeddings: Upload

This notebook uploads embeddings created in a previous step to a vector store for later use. It makes use of [EmbAPI](https://github.com/mpilhlt/embapi/), a vectore store API that has been developed by [Andreas Wagner](https://www.lhlt.mpg.de/wagner/en) for these experiments at the [Max Planck Institute for Legal History and Legal Theory](https://www.lhlt.mpg.de/).

## 0. Preliminaries

In [1]:
# Get info about python version
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

/home/awagner/vcs/digicademy/svsal-bertopic/.venv/bin/python
3.11.14 (main, Dec 17 2025, 21:07:37) [Clang 21.1.4 ]
sys.version_info(major=3, minor=11, micro=14, releaselevel='final', serial=0)


## 1. Setup

### 1.1 Install libraries

Instead of using the below python/ipython commands, and in order to make the notebook more declarative/reproducible, we try to define the necessary libraries and environment in a `uv` *project*, i.e. in the [pyproject.toml file](./pyproject.toml) that controls how `uv` manages the `.venv` virtual environment.

According to the [uv documentation](https://docs.astral.sh/uv/concepts/projects/layout/#the-project-environment):

> To run a command in the project environment, use `uv run`. Alternatively the project environment can be activated as normal for a virtual environment.
>
> When `uv run` is invoked, it will create the project environment if it does not exist yet or ensure it is up-to-date if it exists. The project environment can also be explicitly created with `uv sync`.
>
> It is *not* recommended to modify the project environment manually, e.g., with `uv pip install`. For project dependencies, use `uv add` to add a package to the environment.

### 1.2 Load libraries

In [2]:
import os
import logging
import dotenv
import polars as pl
import json
from tqdm import tqdm
from time import sleep
import numpy as np
import pickle
from typing import Any, Dict, List, Optional
from urllib import parse
import requests
import glob
from pathlib import Path
from collections import deque
from datetime import datetime, timedelta
from email.utils import parsedate_to_datetime

### 1.3 Configuration

#### 1.3.1 API Keys Setup

In [3]:
# Find the .env file and load it
dotenv.load_dotenv(dotenv.find_dotenv())

# Get API keys with fallbacks
api_keys = {
    "embapi": os.getenv("EMBAPI_API_KEY", "")
}
if api_keys["embapi"] == "":
    raise ValueError("API key for EmbAPI (EMBAPI_API_KEY) not found in environment variables.")

print("API keys loaded successfully.")

API keys loaded successfully.


#### 1.3.2 Provider Configuration

In [4]:
# EmbAPI API configuration
API_USER = os.getenv("API_USER", "user")
API_BASE_URL = os.getenv("API_BASE_URL", "https://api.embapi.dev/v1")

# Provider-to-project mapping
# This maps embedding provider names to EmbAPI project names
PROVIDER_PROJECT_MAPPING = {
    "openai_text-embedding-3-large":                   os.getenv("PROJECT_OPENAI_LARGE",    "openai-large"),
    "openai_text-embedding-3-small":                   os.getenv("PROJECT_OPENAI_SMALL",    "openai-small"),
    "academic-cloud_e5-mistral-7b-instruct":           os.getenv("PROJECT_E5_MISTRAL",     "e5-mistral"),
    "academic-cloud_multilingual-e5-large-instruct":   os.getenv("PROJECT_MULTILINGUAL_E5", "multilingual-e5"),
    "academic-cloud_qwen3-embedding-4b":               os.getenv("PROJECT_QWEN3",           "qwen3"),
    "cohere_embed-v4.0_clustering":                    os.getenv("PROJECT_COHERE",          "cohere"),
    "google_gemini-embedding-001_SEMANTIC_SIMILARITY": os.getenv("PROJECT_GEMINI_SIMIL",    "gemini-simil")
}

# Provider-to-LLM instance handle mapping
# This maps embedding provider names to EmbAPI LLM instance handles
PROVIDER_INSTANCE_MAPPING = {
    "openai_text-embedding-3-large":                   os.getenv("INSTANCE_OPENAI_SMALL",    "openai-small"),
    "openai_text-embedding-3-small":                   os.getenv("INSTANCE_OPENAI_SMALL",    "openai-small"),
    "academic-cloud_e5-mistral-7b-instruct":           os.getenv("INSTANCE_E5_MISTRAL",      "ac-e5-mistral"),
    "academic-cloud_multilingual-e5-large-instruct":   os.getenv("INSTANCE_MULTILINGUAL_E5", "ac-multilingual-e5"),
    "academic-cloud_qwen3-embedding-4b":               os.getenv("INSTANCE_QWEN3",           "ac-qwen3"),
    "cohere_embed-v4.0_clustering":                    os.getenv("INSTANCE_COHERE",          "cohere-v4"),
    "google_gemini-embedding-001_SEMANTIC_SIMILARITY": os.getenv("INSTANCE_GEMINI",          "gemini-001")
}

# Instance owner (typically the same as API_USER)
INSTANCE_OWNER = os.getenv("INSTANCE_OWNER", API_USER)

# Instance configurations for creating LLM service instances
# Specifies whether to create from a system definition or standalone
INSTANCE_CONFIGS = {
    "openai_text-embedding-3-large": {
        "definition_owner": "_system",
        "definition_handle": "openai-large",
    },
    "openai_text-embedding-3-small": {
        "definition_owner": "_system",
        "definition_handle": "openai-small",
    },
    "academic-cloud_e5-mistral-7b-instruct": {
        "definition_owner": None,  # Standalone instance
        "definition_handle": None,
    },
    "academic-cloud_multilingual-e5-large-instruct": {
        "definition_owner": None,  # Standalone instance
        "definition_handle": None,
    },
    "academic-cloud_qwen3-embedding-4b": {
        "definition_owner": None,  # Standalone instance
        "definition_handle": None,
    },
    "cohere_embed-v4.0_clustering": {
        "definition_owner": "_system",
        "definition_handle": "cohere-v4",
    },
    "google_gemini-embedding-001_SEMANTIC_SIMILARITY": {
        "definition_owner": "_system",
        "definition_handle": "gemini-embedding-001",
    },
}

# Default API endpoints by provider (used when base_url not in metadata)
DEFAULT_ENDPOINTS = {
    "openai": "https://api.openai.com/v1/embeddings",
    "cohere": "https://api.cohere.ai/v1/embed",
    "google": "https://generativelanguage.googleapis.com/v1beta/models/{model}:embedContent",
    "default": "https://api.example.com/v1/embeddings"
}

# Default configuration (can be overridden per provider)
HEADERS = {"Content-Type": "application/json", "Authorization": f"Bearer {api_keys['embapi']}"}
BATCH_SIZE = None # Set to None to disable batching
RETRY_ATTEMPTS = 3
DELAY_BETWEEN_REQUESTS = 0.2
VERBOSE = False

#### 1.3.3 Throttling Configuration

In order not to put too much stress on the vector database, we can throttle our uploads. Here, we can configure the throttling.

In [5]:
# Choose one of the following throttling strategies, or combine them:

# Option 1: Simple rate limiting (requests per second)
MAX_REQUESTS_PER_SECOND = 5.0  # e.g., 10.0 for 10 requests/sec

# Option 2: Per-minute rate limiting
MAX_REQUESTS_PER_MINUTE = 250  # e.g., 100 for 100 requests/min

# Option 3: Per-hour rate limiting
MAX_REQUESTS_PER_HOUR = None  # e.g., 1000 for 1000 requests/hour

# Option 4: Combine multiple limits (the most restrictive will apply)
# MAX_REQUESTS_PER_SECOND = 5.0    # Max 5 per second
# MAX_REQUESTS_PER_MINUTE = 200    # Max 200 per minute
# MAX_REQUESTS_PER_HOUR = 10000    # Max 10000 per hour

# Advanced options
RESPECT_RETRY_AFTER = True   # Respect Retry-After headers from 429 responses
ADAPTIVE_THROTTLING = True   # Automatically slow down on repeated 429 errors

# Examples of different throttling strategies:
# 
# Conservative (for strict rate limits):
#   MAX_REQUESTS_PER_SECOND = 2.0
#   MAX_REQUESTS_PER_MINUTE = 100
#
# Moderate (balanced approach):
#   MAX_REQUESTS_PER_SECOND = 5.0
#   MAX_REQUESTS_PER_MINUTE = 250
#
# Aggressive (for high-throughput APIs):
#   MAX_REQUESTS_PER_SECOND = 20.0
#   MAX_REQUESTS_PER_MINUTE = 1000
#
# Unknown limits (let adaptive throttling handle it):
#   MAX_REQUESTS_PER_SECOND = None
#   ADAPTIVE_THROTTLING = True
#   RESPECT_RETRY_AFTER = True

#### 1.3.4 File paths

In the following directory, input files will be auto-detected based on the most recent timestamp.

- Priority order: parquet > pickle > csv (for docs)
- For embeddings: parquet > pickle

We'll search for files with this pattern:

- `YYYY-MM-DD_*_docs.{parquet,pkl,csv}`
- `YYYY-MM-DD_*_embeddings.{parquet,pkl}`
- `YYYY-MM-DD_*_processing_metadata.json`
- `YYYY-MM-DD_*_embedding_statistics.json`

In [6]:
# Input files
file_path_in = './out-data' # this was the output path of the previous step, hence the name

#### 1.3.5 Limits

This is used best for checking and improving the workflow with few documents only. Set `max_documents` to `-1` in order to upload all data available in the input files.

In [7]:
# Here we set the number of documents (paragraphs) to process
# Set it to a lower positive integer until everything runs well, then increase it
# Set it to -1 to process all documents:
max_documents=20

## 2. Utility Functions

### 2.1 Logging Configuration

Configure structured logging for the embedding generation process.

In [8]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

## 3. Read Input Data

### 3.1 Load data

In [9]:
def find_manifest_file(directory: str, pattern: str) -> Optional[str]:
    """Find the manifest file matching the pattern."""
    files = glob.glob(os.path.join(directory, pattern))
    if not files:
        return None
    # Sort by modification time, most recent first
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

def find_latest_files(directory: str, pattern: str) -> Optional[str]:
    """Find the most recent file matching the pattern."""
    files = glob.glob(os.path.join(directory, pattern))
    if not files:
        return None
    # Sort by modification time, most recent first
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

def load_docs_data(directory: str) -> pl.DataFrame:
    """Load docs data from parquet, pickle, or CSV, in that order."""
    # Try parquet first
    parquet_file = find_latest_files(directory, "*_docs.parquet")
    if parquet_file:
        print(f"Loading docs from parquet: {parquet_file}")
        return pl.read_parquet(parquet_file, n_rows=max_documents if max_documents > 0 else None)

    # Try pickle
    pickle_file = find_latest_files(directory, "*_docs.pkl")
    if pickle_file:
        print(f"Loading docs from pickle: {pickle_file}")
        with open(pickle_file, "rb") as f:
            df = pickle.load(f)[:max_documents] if max_documents > 0 else pickle.load(f)
            # If it's a pandas DataFrame, convert to polars
            if hasattr(df, 'to_dict'):  # pandas DataFrame check
                return pl.from_pandas(df)
            return df
    
    # Try CSV
    csv_file = find_latest_files(directory, "*_docs.csv")
    if csv_file:
        print(f"Loading docs from CSV: {csv_file}")
        return pl.read_csv(csv_file, n_rows=max_documents if max_documents > 0 else None)
    
    raise FileNotFoundError(f"No docs files found in {directory}")

def load_embeddings_data(directory: str, doc_ids: Optional[set] = None) -> dict:
    """Load embeddings data from parquet or pickle, in that order.
    Returns a nested dict: {provider_name: {doc_id: [embedding_vector]}}
    
    Args:
        directory: Directory containing embeddings files
        doc_ids: Optional set of document IDs to filter embeddings. 
                 If provided, only embeddings for these IDs will be loaded.
    """
    # Try parquet first
    parquet_file = find_latest_files(directory, "*_embeddings.parquet")
    if parquet_file:
        print(f"Loading embeddings from parquet: {parquet_file}")
        # The parquet was created from a nested dict: {provider: {doc_id: embedding}}
        # When saved with pl.DataFrame(nested_dict).write_parquet(), 
        # it creates columns named after providers, with each cell containing the doc_id->embedding mapping
        df_embeddings = pl.read_parquet(parquet_file)
        
        # Convert back to nested dict structure
        # Each column represents a provider, column values are the {doc_id: embedding} dicts
        embeddings_dict = {}
        for column_name in df_embeddings.columns:
            # Get the provider's embeddings (should be a single row with a dict/struct)
            provider_data = df_embeddings[column_name][0]
            if provider_data is not None:
                # Filter embeddings if doc_ids is provided
                if doc_ids is not None:
                    provider_data = {k: v for k, v in provider_data.items() if k in doc_ids}
                embeddings_dict[column_name] = provider_data
        
        if doc_ids is not None:
            print(f"Filtered embeddings to match {len(doc_ids)} loaded documents")
        
        return embeddings_dict
    
    # Try pickle
    pickle_file = find_latest_files(directory, "*_embeddings.pkl")
    if pickle_file:
        print(f"Loading embeddings from pickle: {pickle_file}")
        with open(pickle_file, "rb") as f:
            embeddings_dict = pickle.load(f)
            
        # Filter embeddings if doc_ids is provided
        if doc_ids is not None:
            for provider_name in embeddings_dict:
                embeddings_dict[provider_name] = {
                    k: v for k, v in embeddings_dict[provider_name].items() if k in doc_ids
                }
            print(f"Filtered embeddings to match {len(doc_ids)} loaded documents")
            
        return embeddings_dict
    
    raise FileNotFoundError(f"No embeddings files found in {directory}")

# As a fallback, extract embeddings directly from the docs DataFrame
def extract_embeddings_from_dataframe(docs: pl.DataFrame, provider_name: str) -> dict:
    col_name = f"embeddings_{provider_name}"
    if col_name not in docs.columns:
        raise ValueError(f"Column {col_name} not found in docs DataFrame")
    return {
        row["url"]: row[col_name]
        for row in docs.iter_rows(named=True)
        if row[col_name] is not None
    }

def load_metadata(directory: str) -> dict:
    """Load processing metadata JSON."""
    metadata_file = find_latest_files(directory, "*_processing_metadata.json")
    if metadata_file:
        print(f"Loading metadata from: {metadata_file}")
        with open(metadata_file, "r") as f:
            return json.load(f)
    return {}

# Load all data
print("="*80)

print("Load manifest file")
manifest_file = find_manifest_file(file_path_in, "*_manifest.json")
# Load manifest content
manifest = {}
if manifest_file:
    with open(manifest_file, "r") as f:
        manifest = json.load(f)
    print(f"Loaded manifest from: {manifest_file}")

print("Loading data files...")
docs = load_docs_data(file_path_in)

# Get the set of loaded document IDs to filter embeddings
loaded_doc_ids = set(docs["url"].to_list())

print("Loading embeddings data...")
# load embeddings - preferably from embeddings files, otherwise extract from docs DataFrame
# Pass loaded document IDs to filter embeddings when max_documents is set
embeddings_dict = load_embeddings_data(file_path_in, doc_ids=loaded_doc_ids if max_documents > 0 else None)
if not embeddings_dict:
    embeddings_dict = {}
    for provider in PROVIDER_PROJECT_MAPPING.keys():
        try:
            provider_embeddings = extract_embeddings_from_dataframe(docs, provider)
            embeddings_dict[provider] = provider_embeddings
            print(f"Extracted embeddings for provider {provider} from docs DataFrame.")
        except ValueError as e:
            print(f"Could not extract embeddings for provider {provider}: {e}")

print("Loading metadata...")
metadata = load_metadata(directory=file_path_in)

# Check loaded data and configured maps against manifest (if available)
# Check if for all keys in the manifest's "providers", we have loaded embeddings
# Cancel run if there are discrepancies
if manifest and "providers" in manifest:
    manifest_providers = manifest["providers"].keys()
    loaded_providers = embeddings_dict.keys()
    for provider in manifest_providers:
        if provider not in loaded_providers:
            print(f"Warning: Provider {provider} listed in manifest but no embeddings loaded.")
            raise ValueError(f"Embeddings for provider {provider} missing.")

    for provider in loaded_providers:
        if provider not in manifest_providers:
            print(f"Warning: Embeddings loaded for provider {provider} not listed in manifest.")
            raise ValueError(f"Embeddings for provider {provider} not in manifest.")

# Check if all providers in embeddings_dict have a mapping in PROVIDER_PROJECT_MAPPING
# Cancel run if there are discrepancies
for provider in embeddings_dict.keys():
    if provider not in PROVIDER_PROJECT_MAPPING:
        print(f"Warning: No project mapping found for provider {provider}.")
        raise ValueError(f"Project mapping for provider {provider} missing.")

print(f"\nLoaded {docs.height} documents")
print(f"Loaded embeddings for {len(embeddings_dict)} providers:")
for provider, provider_embeddings in embeddings_dict.items():
    print(f"  - {provider}: {len(provider_embeddings)} documents")

if metadata:
    print(f"\nProcessing metadata:")
    print(f"  Processing date: {metadata.get('processing_date', 'N/A')}")
    if 'providers' in metadata:
        print(f"  Providers in metadata: {list(metadata['providers'].keys())}")


Load manifest file
Loaded manifest from: ./out-data/embeddings_manifest.json
Loading data files...
Loading docs from parquet: ./out-data/2026-01-28_all_docs.parquet
Loading embeddings data...
Loading embeddings from parquet: ./out-data/2026-01-28_all_embeddings.parquet
Filtered embeddings to match 20 loaded documents
Loading metadata...
Loading metadata from: ./out-data/2026-01-28_all_processing_metadata.json

Loaded 20 documents
Loaded embeddings for 3 providers:
  - openai_text-embedding-3-small: 20 documents
  - cohere_embed-v4.0_clustering: 20 documents
  - google_gemini-embedding-001_SEMANTIC_SIMILARITY: 20 documents

Processing metadata:
  Processing date: 2026-01-28 17:53:56
  Providers in metadata: ['openai_text-embedding-3-small', 'cohere_embed-v4.0_clustering', 'google_gemini-embedding-001_SEMANTIC_SIMILARITY']


### 3.2 Give some information about the data

In [10]:
print("="*80)
print("Data Overview")
print("="*80)
print(f"\nShape of docs dataframe: {docs.shape}")
print(f"Number of available documents: {docs.height}")
print(f"\nDocument length statistics:")
if "len_content" in docs.columns:
    print(docs["len_content"].describe())
else:
    print("  (len_content column not available)")

print("\nEmbeddings by provider:")
for provider, provider_embeddings in embeddings_dict.items():
    print(f"  {provider}: {len(provider_embeddings)} embeddings")
    # Show embedding dimension
    if provider_embeddings:
        sample_embedding = next(iter(provider_embeddings.values()))
        print(f"    - Dimension: {len(sample_embedding)}")

print("\nFirst 3 rows of docs dataframe:")
docs.head(3)

Data Overview

Shape of docs dataframe: (20, 14)
Number of available documents: 20

Document length statistics:
  (len_content column not available)

Embeddings by provider:
  openai_text-embedding-3-small: 20 embeddings
    - Dimension: 1536
  cohere_embed-v4.0_clustering: 20 embeddings
    - Dimension: 1536
  google_gemini-embedding-001_SEMANTIC_SIMILARITY: 20 embeddings
    - Dimension: 1536

First 3 rows of docs dataframe:


url,xmlid,lang,wid,author-id,author-name,title,year,passage,citation-recommendation,content,embeddings_openai_text-embedding-3-small,embeddings_cohere_embed-v4.0_clustering,embeddings_google_gemini-embedding-001_SEMANTIC_SIMILARITY
str,str,str,str,str,str,str,i64,str,str,str,list[f32],list[f32],list[f32]
"""https://id.salamanca.school/te…","""W0001-01-0006-tp-03e8""","""la""","""W0001""","""A0007""","""Avendaño""","""Thesaurus Indicus""",1668,"""vol. 1""","""Avendaño, Thesaurus Indicus (2…","""R. P. DIDACI DE AVENDA N O SOC…","[0.021436, -0.01359, … -0.021916]","[-39.0, -1.0, … 29.0]","[-0.024484, -0.033015, … 0.014212]"
"""https://id.salamanca.school/te…","""W0001-01-0008-he-03e8""","""la""","""W0001""","""A0007""","""Avendaño""","""Thesaurus Indicus""",1668,"""vol. 1 praef. 1""","""Avendaño, Thesaurus Indicus (2…","""REGIO INDIARVM CONSILIO inter …","[0.00497, 0.012981, … 0.022147]","[-35.0, -22.0, … 57.0]","[-0.032382, -0.018333, … 0.010588]"
"""https://id.salamanca.school/te…","""W0001-01-0008-pa-03eb""","""la""","""W0001""","""A0007""","""Avendaño""","""Thesaurus Indicus""",1668,"""vol. 1 praef. 1 paragr. 'THesa…","""Avendaño, Thesaurus Indicus (2…","""THesavrvs Indicvs Regij sane i…","[0.034662, -0.01489, … -0.023151]","[-77.0, 23.0, … 33.0]","[-0.037139, -0.043705, … 0.01199]"


### 3.3 Data Structure Overview

The loaded data has the following structure:
- **docs**: Polars DataFrame with document metadata (url, content, xmlid, author, etc.)
- **embeddings_dict**: Nested dictionary organized as:
  ```python
  {
      "provider_name_1": {
          "doc_id_1": [embedding_vector_1],
          "doc_id_2": [embedding_vector_2],
          ...
      },
      "provider_name_2": {
          "doc_id_1": [embedding_vector_1],
          ...
      }
  }
  ```
- **metadata**: Configuration and processing information from the embedding creation run

## 4. Uploading to Vector Store

Define functions to build uploadable data either from dataframe plus embeddings, or from dataframe including embeddings.

In [11]:
def build_json_object(
    doc_id: str, 
    doc_row: dict, 
    embedding: list, 
    provider_name: str,
    api_user: str,
    api_project: str,
    instance_owner: str,
    instance_handle: str
) -> dict:
    """Build a JSON object for a single document embedding for EmbAPI upload.
    
    Dynamically includes all fields from doc_row in metadata except those 
    used at the top level (content -> text).
    """
    
    # Fields that are used at the top level and should not be duplicated in metadata
    excluded_fields = {
        "content",  # Used as "text" at top level
    }
    
    # Build metadata by including all doc_row fields except excluded ones
    metadata = {}
    
    for key, value in doc_row.items():
        # Skip excluded fields
        if key in excluded_fields:
            continue
        
        # Skip fields that look like embeddings (large arrays/lists)
        # Embeddings are typically 100+ dimensions
        if isinstance(value, (list, np.ndarray)) and len(value) > 50:
            continue
        
        # Convert numpy types to Python native types for JSON serialization
        if isinstance(value, np.integer):
            metadata[key] = int(value)
        elif isinstance(value, np.floating):
            metadata[key] = float(value)
        elif isinstance(value, np.ndarray):
            metadata[key] = value.tolist()
        else:
            metadata[key] = value
    
    # Build the json object with instance_owner and instance_handle
    json_object = {
        "text_id": parse.quote(doc_id, safe=''),
        "user_handle": api_user,
        "project_handle": api_project,
        "instance_owner": instance_owner,
        "instance_handle": instance_handle,
        "text": doc_row["content"],
        "vector": embedding,
        "vector_dim": len(embedding),
        "metadata": metadata
    }
    return json_object

def build_json_objects_for_provider(
    provider_name: str,
    docs_df: pl.DataFrame,
    embeddings: dict,
    api_user: str,
    api_project: str,
    instance_owner: str,
    instance_handle: str,
    max_objects: Optional[int] = None
) -> list:
    """Build JSON objects for all documents with embeddings from a specific provider.
    
    Args:
        provider_name: Name of the embedding provider
        docs_df: Polars DataFrame with document data
        embeddings: Dict mapping doc_id to embedding vector
        api_user: EmbAPI user handle
        api_project: EmbAPI project handle
        instance_owner: LLM instance owner
        instance_handle: LLM instance handle
        max_objects: Optional maximum number of JSON objects to build (for testing)
    Returns:
        List of JSON objects ready for EmbAPI upload
    """
    json_objects = []
    
    # Create a lookup dict from the dataframe for faster access
    docs_dict = {row["url"]: row for row in docs_df.iter_rows(named=True)}
    
    # Iterate over the embeddings for this provider
    for doc_id, embedding in tqdm(embeddings.items(), desc=f"Building payloads for {provider_name}"):
        if max_objects and len(json_objects) >= max_objects:
            print(f"Reached max_objects limit of {max_objects} for provider {provider_name}")
            break

        if doc_id not in docs_dict:
            print(f"Warning: Document {doc_id} not found in docs dataframe")
            continue
        
        if not embedding or len(embedding) == 0:
            print(f"Warning: Empty embedding for document {doc_id}")
            continue
        
        doc_row = docs_dict[doc_id]
        json_obj = build_json_object(
            doc_id=doc_id,
            doc_row=doc_row,
            embedding=embedding,
            provider_name=provider_name,
            api_user=api_user,
            api_project=api_project,
            instance_owner=instance_owner,
            instance_handle=instance_handle
        )
        json_objects.append(json_obj)
    
    print(f"Built {len(json_objects)} JSON objects for {provider_name}")
    return json_objects

def build_all_json_objects(
    docs_df: pl.DataFrame,
    embeddings_dict: dict,
    provider_project_mapping: dict,
    provider_instance_mapping: dict,
    instance_owner: str,
    api_user: str,
    max_objects: Optional[int] = None
) -> dict:
    """Build JSON objects for all providers.
    
    Args:
        docs_df: Polars DataFrame with document data
        embeddings_dict: Nested dict {provider: {doc_id: embedding}}
        provider_project_mapping: Dict mapping provider names to EmbAPI project names
        provider_instance_mapping: Dict mapping provider names to LLM instance handles
        instance_owner: LLM instance owner (typically same as api_user)
        api_user: EmbAPI user handle
        max_objects: Optional maximum number of JSON objects to build per provider (for testing)
    Returns:
        Dict mapping provider names to lists of JSON objects
    """
    all_json_objects = {}
    
    for provider_name, provider_embeddings in embeddings_dict.items():
        if provider_name not in provider_project_mapping:
            print(f"Warning: No project mapping for provider '{provider_name}', skipping")
            continue
        
        if provider_name not in provider_instance_mapping:
            print(f"Warning: No instance mapping for provider '{provider_name}', skipping")
            continue
        
        project_name = provider_project_mapping[provider_name]
        instance_handle = provider_instance_mapping[provider_name]
        
        json_objects = build_json_objects_for_provider(
            provider_name=provider_name,
            docs_df=docs_df,
            embeddings=provider_embeddings,
            api_user=api_user,
            api_project=project_name,
            instance_owner=instance_owner,
            instance_handle=instance_handle,
            max_objects=max_objects if max_objects else None
        )
        
        all_json_objects[provider_name] = json_objects
    
    return all_json_objects

def build_instance_configs(
    embeddings_dict: dict,
    metadata: dict,
    provider_instance_mapping: dict,
    instance_configs: dict
) -> dict:
    """Build LLM instance configurations from metadata and embeddings.
    
    Args:
        embeddings_dict: Nested dict {provider: {doc_id: embedding}}
        metadata: Processing metadata with provider configurations
        provider_instance_mapping: Dict mapping provider names to LLM instance handles
        instance_configs: Dict specifying definition_owner/definition_handle for each provider
        
    Returns:
        Dict mapping provider names to LLM instance configurations
    """
    llm_instance_configs = {}
    
    for provider_name in embeddings_dict.keys():
        if provider_name not in provider_instance_mapping:
            continue
        
        # Get provider metadata
        provider_meta = metadata.get("providers", {}).get(provider_name, {})
        
        # Get actual dimensions from first embedding
        provider_embeddings = embeddings_dict[provider_name]
        actual_dimensions = None
        if provider_embeddings:
            first_embedding = next(iter(provider_embeddings.values()))
            actual_dimensions = len(first_embedding)
        
        # Build endpoint URL
        base_url = provider_meta.get("base_url")
        model = provider_meta.get("model", "unknown")
        provider = provider_meta.get("provider", "")
        
        if base_url:
            endpoint = f"{base_url}/embeddings"
        else:
            # Use configured default endpoints by provider
            if provider in DEFAULT_ENDPOINTS:
                endpoint_template = DEFAULT_ENDPOINTS[provider]
                # Replace {model} placeholder if present
                endpoint = endpoint_template.format(model=model)
            else:
                endpoint = DEFAULT_ENDPOINTS["default"]
        
        # Build description
        model = provider_meta.get("model", "unknown")
        provider_display = provider_meta.get("provider", "unknown")
        description = f"{provider_display} {model} embedding service"
        
        # Get dimensions (prefer actual, fallback to metadata)
        dimensions = actual_dimensions or provider_meta.get("dimensions") or 1536
        
        # Get API standard (use "gemini" for google provider)
        api_standard = provider_meta.get("api_spec", "openai")
        if provider == "google":
            api_standard = "gemini"
        
        # Get instance handle
        instance_handle = provider_instance_mapping[provider_name]
        
        # Get definition info from instance_configs
        config = instance_configs.get(provider_name, {})
        definition_owner = config.get("definition_owner")
        definition_handle = config.get("definition_handle")
        
        # Build instance configuration
        instance_config = {
            "instance_handle": instance_handle,
            "endpoint": endpoint,
            "description": description,
            "api_key": "dummy-key-placeholder",
            "api_standard": api_standard,
            "model": model,
            "dimensions": dimensions
        }
        
        # Add definition references if specified
        if definition_owner and definition_handle:
            instance_config["definition_owner"] = definition_owner
            instance_config["definition_handle"] = definition_handle
        
        llm_instance_configs[provider_name] = instance_config
        
        print(f"LLM instance config for {provider_name}:")
        print(f"  Handle: {instance_handle}, Model: {model}, Dimensions: {dimensions}, API: {api_standard}")
        if definition_owner and definition_handle:
            print(f"  Definition: {definition_owner}/{definition_handle}")
    
    return llm_instance_configs

Next, define a function to upload data to our vector store.

In [12]:
class JsonListApiSender:

    def __init__(
        self,
        api_user: str,
        api_base_url: str,
        headers: Optional[Dict[str, str]] = None,
        retry_attempts: int = 3,
        delay_between_requests: float = 0.1,
        verbose: bool = True,
        # Throttling options
        max_requests_per_second: Optional[float] = None,
        max_requests_per_minute: Optional[int] = None,
        max_requests_per_hour: Optional[int] = None,
        respect_retry_after: bool = True,
        adaptive_throttling: bool = True
    ):
        """Initialize the JSON List API Sender with advanced throttling capabilities.

            Args:
                api_user: User account of the EmbAPI database
                api_base_url: Base URL for the EmbAPI (e.g., 'https://example.com/embapi/v1/embeddings')
                headers: Optional headers for the request
                retry_attempts: Number of times to retry a failed request
                delay_between_requests: Minimum delay between each request (seconds)
                verbose: Whether to print detailed logging
            Throttling Options:
                max_requests_per_second: Maximum requests per second (e.g., 10.0)
                max_requests_per_minute: Maximum requests per minute (e.g., 100)
                max_requests_per_hour: Maximum requests per hour (e.g., 1000)
                respect_retry_after: Whether to respect Retry-After headers from 429 responses
                adaptive_throttling: Automatically slow down on repeated 429 errors
            Examples:
                # Simple rate limiting: 5 requests per second
                sender = JsonListApiSender(..., max_requests_per_second=5.0)
                # Conservative: 100 requests per minute
                sender = JsonListApiSender(..., max_requests_per_minute=100)
                # Multiple limits: 10/sec, 500/min, 10000/hour
                sender = JsonListApiSender(...,
                    max_requests_per_second=10.0,
                    max_requests_per_minute=500,
                    max_requests_per_hour=10000)
        """
        self.api_user = api_user
        self.api_base_url = api_base_url
        self.headers = headers or {"Content-Type": "application/json"}
        self.retry_attempts = retry_attempts
        self.delay_between_requests = delay_between_requests
        self.verbose = verbose
        self.logger = logging.getLogger(__name__)
        # Throttling configuration
        self.max_requests_per_second = max_requests_per_second
        self.max_requests_per_minute = max_requests_per_minute
        self.max_requests_per_hour = max_requests_per_hour
        self.respect_retry_after = respect_retry_after
        self.adaptive_throttling = adaptive_throttling
        # Request tracking for rate limiting
        self.request_timestamps = deque()
        self.last_request_time = None
        self.adaptive_delay = 0.0  # Additional delay from adaptive throttling
        self.consecutive_429s = 0
        # Upload statistics tracking
        self.upload_stats = {
            'total_requests': 0,
            'successful_requests': 0,
            'failed_requests': 0,
            'error_codes': {},  # {status_code: count}
            'request_times': [],  # List of request durations
            'start_time': None,
            'end_time': None
        }
        # Base URLs for different API resources (derive from embeddings base URL)
        self.projects_base_url = api_base_url.replace('/embeddings', '/projects')
        self.llm_instances_base_url = api_base_url.replace('/embeddings', '/llm-instances')
        self.llm_definitions_base_url = api_base_url.replace('/embeddings', '/llm-definitions')

    def _parse_provider_info(self, provider_name: str) -> tuple:
        """Parse provider name to extract platform/provider and model information.

            Args:
                provider_name: Provider string (e.g., "openai_text-embedding-3-small")
            Returns:
                Tuple of (provider, model) strings
        """
        if '_' in provider_name:
            parts = provider_name.split('_', 1)
            provider = parts[0]
            model = parts[1] if len(parts) > 1 else "unknown"
        else:
            provider = provider_name
            model = "unknown"
            
        # Clean up provider name for display
        provider_display = provider.replace('-', ' ').title()
        return provider_display, model

    def check_project_exists(self, project_name: str) -> Optional[dict]:
        """Check if a EmbAPI project exists.

            Args:
                project_name: The project handle to check
            Returns:
                Project info dict if exists, None otherwise
        """
        url = f"{self.projects_base_url}/{self.api_user}/{project_name}"
        try:
            response = requests.get(url, headers=self.headers)
            if response.status_code == 200:
                project_info = response.json()
                if self.verbose:
                    print(f"✓ Project '{project_name}' exists (ID: {project_info.get('project_id')})")
                return project_info
            elif response.status_code == 404:
                if self.verbose:
                    print(f"✗ Project '{project_name}' does not exist")
                return None
            else:
                self.logger.warning(f"Unexpected status {response.status_code} checking project '{project_name}'")
            return None
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error checking project existence: {e}")
            return None

    def create_project(self, project_name: str, provider_name: str, instance_owner: str, instance_handle: str) -> Optional[dict]:
        """Create a new EmbAPI project.

            Args:
                project_name: The project handle to create
                provider_name: The provider name (used to generate description)
                instance_owner: The LLM instance owner
                instance_handle: The LLM instance handle
            Returns:
                Creation response dict if successful, None otherwise
        """
        url = f"{self.projects_base_url}/{self.api_user}"

        # Parse provider info for description
        provider, model = self._parse_provider_info(provider_name)
        description = f"Embeddings from {provider} provider with {model} model"
        payload = {
            "project_handle": project_name,
            "description": description,
            "instance_owner": instance_owner,
            "instance_handle": instance_handle
        }

        try:
            response = requests.post(url, json=payload, headers=self.headers)
            response.raise_for_status()
            result = response.json()
            print(f"✓ Created project '{project_name}' (ID: {result.get('project_id')})")
            print(f"  Description: {description}")
            print(f"  Instance: {instance_owner}/{instance_handle}")
            return result

        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error creating project '{project_name}': {e}")
            if hasattr(e, 'response') and e.response is not None:
                self.logger.error(f"Response: {e.response.content}")
            return None

    def ensure_project_exists(self, project_name: str, provider_name: str, instance_owner: str, instance_handle: str) -> bool:
        """Ensure a project exists, creating it if necessary.

            Args:
                project_name: The project handle
                provider_name: The provider name (for description if creating)
                instance_owner: The LLM instance owner
                instance_handle: The LLM instance handle
            Returns:
                True if project exists or was created successfully, False otherwise
        """
        # Check if project exists
        project_info = self.check_project_exists(project_name)
        if project_info is not None:
            return True

        # Project doesn't exist, create it
        print(f"Creating project '{project_name}'...")
        result = self.create_project(project_name, provider_name, instance_owner, instance_handle)
        return result is not None

    def check_llm_instance_exists(self, instance_handle: str) -> Optional[dict]:
        """Check if an LLM instance exists.

            Args:
                instance_handle: The LLM instance handle to check
            Returns:
                LLM instance info dict if exists, None otherwise
        """
        url = f"{self.llm_instances_base_url}/{self.api_user}/{instance_handle}"
        try:
            response = requests.get(url, headers=self.headers)
            if response.status_code == 200:
                instance_info = response.json()
                if self.verbose:
                    print(f"✓ LLM instance '{instance_handle}' exists (ID: {instance_info.get('llm_instance_id')})")
                return instance_info
            elif response.status_code == 404:
                if self.verbose:
                    print(f"✗ LLM instance '{instance_handle}' does not exist")
                return None
            else:
                self.logger.warning(f"Unexpected status {response.status_code} checking LLM instance '{instance_handle}'")
                return None

        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error checking LLM instance existence: {e}")
            return None

    def create_llm_instance(self, instance_handle: str, instance_config: dict) -> Optional[dict]:
        """Create a new LLM instance.

            Args:
                instance_handle: The LLM instance handle to create
                instance_config: Configuration dict with endpoint, model, dimensions, etc.
            Returns:
                Creation response dict if successful, None otherwise
        """
        url = f"{self.llm_instances_base_url}/{self.api_user}/{instance_handle}"

        # Build the InstanceInput payload
        payload = {
            "instance_handle": instance_handle,
            "endpoint": instance_config.get("endpoint", "https://api.example.com/v1/embeddings"),
            "description": instance_config.get("description", f"LLM instance for {instance_handle}"),
            "api_standard": instance_config.get("api_standard", "openai"),
            "model": instance_config.get("model", "unknown"),
            "dimensions": instance_config.get("dimensions", 1536)
        }

        # Add optional api_key if provided
        if "api_key" in instance_config:
            payload["api_key"] = instance_config["api_key"]

        # Add definition references if provided (for creating from a definition)
        if "definition_owner" in instance_config and instance_config["definition_owner"]:
            payload["definition_owner"] = instance_config["definition_owner"]
        if "definition_handle" in instance_config and instance_config["definition_handle"]:
            payload["definition_handle"] = instance_config["definition_handle"]

        try:
            # Use PUT instead of POST (instance handle is in URL path)
            response = requests.put(url, json=payload, headers=self.headers)
            response.raise_for_status()
            result = response.json()
            print(f"✓ Created LLM instance '{instance_handle}' (ID: {result.get('llm_instance_id')})")
            print(f"  Model: {payload['model']}, Dimensions: {payload['dimensions']}")
            if "definition_owner" in payload:
                print(f"  From definition: {payload.get('definition_owner')}/{payload.get('definition_handle')}")
            return result

        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error creating LLM instance '{instance_handle}': {e}")
            if hasattr(e, 'response') and e.response is not None:
                self.logger.error(f"Response: {e.response.content}")
            return None

    def ensure_llm_instance_exists(self, instance_handle: str, instance_config: dict) -> bool:
        """Ensure an LLM instance exists, creating it if necessary.

            Args:
                instance_handle: The LLM instance handle
                instance_config: Configuration for the instance if it needs to be created
            Returns:
                True if instance exists or was created successfully, False otherwise
        """
        # Check if instance exists
        instance_info = self.check_llm_instance_exists(instance_handle)
        if instance_info is not None:
            return True

        # Instance doesn't exist, create it
        print(f"Creating LLM instance '{instance_handle}'...")
        result = self.create_llm_instance(instance_handle, instance_config)
        return result is not None

    def _wait_for_rate_limit(self):
        """Wait as needed to respect all configured rate limits.
            Uses a sliding window approach for per-minute and per-hour limits.
        """
        now = datetime.now()

        # Clean up old timestamps outside our tracking windows
        cutoff_time = now - timedelta(hours=1)
        while self.request_timestamps and self.request_timestamps[0] < cutoff_time:
            self.request_timestamps.popleft()

        # Check per-second rate limit
        if self.max_requests_per_second:
            if self.last_request_time:
                min_interval = 1.0 / self.max_requests_per_second
                elapsed = (now - self.last_request_time).total_seconds()
                if elapsed < min_interval:
                    wait_time = min_interval - elapsed
                    if self.verbose:
                        print(f"Rate limit: waiting {wait_time:.3f}s (max {self.max_requests_per_second}/sec)")
                    sleep(wait_time)
                    now = datetime.now()

        # Check per-minute rate limit
        if self.max_requests_per_minute:
            minute_ago = now - timedelta(minutes=1)
            recent_requests = sum(1 for ts in self.request_timestamps if ts > minute_ago)
            if recent_requests >= self.max_requests_per_minute:
                # Wait until the oldest request in this minute expires
                wait_until = self.request_timestamps[0] + timedelta(minutes=1)
                wait_time = (wait_until - now).total_seconds() + 0.1
                if wait_time > 0:
                    if self.verbose:
                        print(f"Rate limit: waiting {wait_time:.1f}s (max {self.max_requests_per_minute}/min)")
                    sleep(wait_time)
                    now = datetime.now()

        # Check per-hour rate limit
        if self.max_requests_per_hour:
            hour_ago = now - timedelta(hours=1)
            recent_requests = sum(1 for ts in self.request_timestamps if ts > hour_ago)
            if recent_requests >= self.max_requests_per_hour:
                # Wait until the oldest request in this hour expires
                wait_until = self.request_timestamps[0] + timedelta(hours=1)
                wait_time = (wait_until - now).total_seconds() + 0.1
                if wait_time > 0:
                    print(f"Rate limit: waiting {wait_time:.1f}s (max {self.max_requests_per_hour}/hour)")
                    sleep(wait_time)
                    now = datetime.now()

        # Apply base delay between requests
        if self.delay_between_requests > 0:
            sleep(self.delay_between_requests)

        # Apply adaptive delay if enabled
        if self.adaptive_delay > 0:
            if self.verbose:
                print(f"Adaptive throttling: additional {self.adaptive_delay:.2f}s delay")
            sleep(self.adaptive_delay)

        # Record this request timestamp
        self.last_request_time = datetime.now()
        self.request_timestamps.append(self.last_request_time)

    def send_single_item(self, item: Dict[str, Any], api_endpoint: str) -> bool:
        """Send a single JSON item to the API with retry logic and advanced throttling.

            Args:
                item: JSON object to send
                api_endpoint: Full API endpoint URL for this upload
            Returns:
                Boolean indicating success or failure
        """
        request_start = datetime.now()
        self.upload_stats['total_requests'] += 1
        for attempt in range(self.retry_attempts):
            try:
                # Wait for rate limiting before making request
                self._wait_for_rate_limit()
                response = requests.post(
                    api_endpoint,
                    json={"embeddings": [item]},
                    headers=self.headers
                )

                # Handle 429 Too Many Requests
                if response.status_code == 429:
                    self.consecutive_429s += 1

                    # Check for Retry-After header
                    if self.respect_retry_after and 'Retry-After' in response.headers:
                        retry_after = response.headers['Retry-After']
                        try:
                            # Retry-After can be seconds or HTTP-date
                            wait_time = int(retry_after)
                        except ValueError:
                            # If it's a date, calculate seconds from now
                            retry_date = parsedate_to_datetime(retry_after)
                            wait_time = (retry_date - datetime.now()).total_seconds()

                        print(f"429 Too Many Requests - Retry-After: {wait_time}s")
                        sleep(wait_time)

                    else:
                        # Exponential backoff for 429s
                        backoff_time = min(2 ** self.consecutive_429s, 60)
                        print(f"429 Too Many Requests - backing off {backoff_time}s")
                        sleep(backoff_time)

                    # Apply adaptive throttling
                    if self.adaptive_throttling:
                        self.adaptive_delay = min(self.adaptive_delay + 0.5, 5.0)
                        print(f"Adaptive throttling increased to {self.adaptive_delay}s")

                    # Don't count 429 as an attempt if we're retrying
                    continue

                response.raise_for_status()

                # Track successful request
                request_duration = (datetime.now() - request_start).total_seconds()
                self.upload_stats['successful_requests'] += 1
                self.upload_stats['request_times'].append(request_duration)

                # Reset consecutive 429 counter on success
                if self.consecutive_429s > 0:
                    self.consecutive_429s = 0

                    # Gradually reduce adaptive delay on success
                    if self.adaptive_throttling and self.adaptive_delay > 0:
                        self.adaptive_delay = max(self.adaptive_delay - 0.1, 0.0)

                if self.verbose:
                    print(f"Successfully sent item with text_id: {item.get('text_id')}")

                return True

            except requests.exceptions.RequestException as e:
                # Track error code if available
                if hasattr(e, 'response') and e.response is not None:
                    status_code = e.response.status_code
                    self.upload_stats['error_codes'][status_code] = self.upload_stats['error_codes'].get(status_code, 0) + 1

                    # Log error only for the last attempt
                    error_msg = ""
                    if attempt == self.retry_attempts - 1:
                        error_msg = str(e)

                    if hasattr(e, 'response') and e.response is not None:
                        self.logger.error(f"Failed to send item. Error: {error_msg}\nResponse: {e.response.content}")
                    else:
                        self.logger.error(f"Failed to send item. Error: {error_msg}")
                        
                    # Exponential backoff
                    sleep(2 ** attempt)

        # If we exhausted all retries
        self.upload_stats['failed_requests'] += 1
        return False

    def send_list(
        self,
        json_list: List[Dict[str, Any]],
        api_project: str,
        num_items: Optional[int] = None
    ) -> Dict[str, Any]:
        """Send a list of JSON objects to the API for a specific project.

            Args:
                json_list: List of JSON objects to send
                api_project: EmbAPI project handle
                num_items: Number of items to send (None = send all)
            Returns:
                Dictionary with counts of successful and failed items
        """
        # Build API endpoint for this project
        api_endpoint = f"{self.api_base_url}/{self.api_user}/{api_project}"

        # Determine number of items to send
        if num_items is None:
            num_items = len(json_list)
        else:
            num_items = min(num_items, len(json_list))

        # Slice the list to the desired number of items
        items_to_send = json_list[:num_items]

        # Tracking variables
        success_count = 0
        fail_count = 0
        failed_items = []

        # Track upload timing
        upload_start_time = datetime.now()
        if self.upload_stats['start_time'] is None:
            self.upload_stats['start_time'] = upload_start_time

            # Log start of sending process
            self.logger.info(f"Starting to send {num_items} items to {api_endpoint}")
            print(f"\nUploading {num_items} items to project '{api_project}'...")

        # Show throttling configuration
        throttling_info = []
        if self.max_requests_per_second:
            throttling_info.append(f"{self.max_requests_per_second}/sec")

        if self.max_requests_per_minute:
            throttling_info.append(f"{self.max_requests_per_minute}/min")

        if self.max_requests_per_hour:
            throttling_info.append(f"{self.max_requests_per_hour}/hour")

        if throttling_info:
            print(f"Rate limits: {', '.join(throttling_info)}")

        # Progress bar
        for item in tqdm(items_to_send, desc=f"Uploading to {api_project}", total=num_items):
            if self.send_single_item(item, api_endpoint):
                success_count += 1
            else:
                fail_count += 1
                failed_items.append(item.get('text_id'))

        # Track end time
        upload_end_time = datetime.now()
        self.upload_stats['end_time'] = upload_end_time
        upload_duration = (upload_end_time - upload_start_time).total_seconds()

        # Log overall results
        self.logger.info(
            f"Upload complete for {api_project}. "
            f"Sent: {num_items}, "
            f"Successful: {success_count}, "
            f"Failed: {fail_count}"
        )
        
        return {
            "project": api_project,
            "total_sent": num_items,
            "successful": success_count,
            "failed": fail_count,
            "failed_ids": failed_items,
            "upload_duration_seconds": upload_duration
        }

    def send_all_providers(
        self,
        provider_json_objects: Dict[str, List[Dict[str, Any]]],
        provider_project_mapping: Dict[str, str],
        provider_instance_mapping: Dict[str, str],
        instance_owner: str,
        instance_configs: Dict[str, dict],
        num_items: Optional[int] = None
    ) -> Dict[str, Dict[str, Any]]:
        """Send JSON objects for all providers to their respective EmbAPI projects.

            Args:
                provider_json_objects: Dict mapping provider names to lists of JSON objects
                provider_project_mapping: Dict mapping provider names to EmbAPI project names
                provider_instance_mapping: Dict mapping provider names to LLM instance handles
                instance_owner: LLM instance owner
                instance_configs: Dict mapping provider names to LLM instance configurations
                num_items: Number of items to send per provider (None = send all)
            Returns:
                Dict mapping provider names to their upload results
        """
        results = {}
        print("="*80)
        print("Starting multi-provider upload")
        print("="*80)
        for provider_name, json_objects in provider_json_objects.items():
            if provider_name not in provider_project_mapping:
                print(f"Warning: No project mapping for provider '{provider_name}', skipping")
                continue

            if provider_name not in provider_instance_mapping:
                print(f"Warning: No instance mapping for provider '{provider_name}', skipping")
                continue

            project_name = provider_project_mapping[provider_name]
            instance_handle = provider_instance_mapping[provider_name]

            print(f"{provider_name} -> {project_name}")
            print(f"  Items to upload: {len(json_objects)}")

            # Ensure LLM instance exists BEFORE project (project requires valid instance)
            instance_config = instance_configs.get(provider_name, {})
            if not self.ensure_llm_instance_exists(instance_handle, instance_config):
                print(f"  ✗ Failed to ensure LLM instance exists, skipping upload")
                results[provider_name] = {
                    "project": project_name,
                    "total_sent": 0,
                    "successful": 0,
                    "failed": 0,
                    "failed_ids": [],
                    "error": "LLM instance does not exist and could not be created"
                }
                continue

            # Ensure project exists (now that instance exists)
            if not self.ensure_project_exists(project_name, provider_name, instance_owner, instance_handle):
                print(f"  ✗ Failed to ensure project exists, skipping upload")
                results[provider_name] = {
                    "project": project_name,
                    "total_sent": 0,
                    "successful": 0,
                    "failed": 0,
                    "failed_ids": [],
                    "error": "Project does not exist and could not be created"
                }
                continue

            result = self.send_list(
                json_list=json_objects,
                api_project=project_name,
                num_items=num_items
            )
            results[provider_name] = result
            print(f"  ✓ Successful: {result['successful']}")
            if result['failed'] > 0:
                print(f"  ✗ Failed: {result['failed']}")

        print("" + "="*80)
        print("Upload Summary")
        print("="*80)
        for provider_name, result in results.items():
            print(f"{provider_name}:")
            print(f"  Total: {result['total_sent']}, Success: {result['successful']}, Failed: {result['failed']}")

        return results

    def get_all_projects(self) -> Optional[dict]:
        """Get all EmbAPI projects for the user.

            Returns:
                API response with all projects, or None on error
        """
        url = f"{self.projects_base_url}/{self.api_user}"
        try:
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error fetching projects: {e}")
            return None

    def get_project_info(self, project_name: str) -> Optional[dict]:
        """Get detailed information about a single EmbAPI project.
        
            Args:
                project_name: The project handle to query
            Returns:
                Project info dict with detailed information including embedding count, or None on error
        """
        url = f"{self.projects_base_url}/{self.api_user}/{project_name}"
        try:
            response = requests.get(url, headers=self.headers)
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 404:
                self.logger.warning(f"Project '{project_name}' not found")
                return None
            else:
                self.logger.warning(f"Unexpected status {response.status_code} for project '{project_name}'")
                return None
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error fetching project info for '{project_name}': {e}")
            return None


    def get_all_llm_instances(self) -> Optional[dict]:
        """Get all EmbAPI LLM service instances for the user.

            Returns:
                API response with all LLM service instances, or None on error
        """
        url = f"{self.llm_instances_base_url}/{self.api_user}"
        try:
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error fetching LLM instances: {e}")
            return None

    def get_upload_statistics(self) -> dict:
        """Get comprehensive upload statistics.

            Returns:
                Dictionary with detailed upload statistics
        """
        stats = self.upload_stats.copy()

        # Calculate derived statistics
        if stats['request_times']:
            stats['avg_request_time'] = sum(stats['request_times']) / len(stats['request_times'])
            stats['max_request_time'] = max(stats['request_times'])
            stats['min_request_time'] = min(stats['request_times'])
            stats['total_request_time'] = sum(stats['request_times'])
        else:
            stats['avg_request_time'] = 0
            stats['max_request_time'] = 0
            stats['min_request_time'] = 0
            stats['total_request_time'] = 0

        # Calculate total elapsed time
        if stats['start_time'] and stats['end_time']:
            stats['total_elapsed_time'] = (stats['end_time'] - stats['start_time']).total_seconds()
        else:
            stats['total_elapsed_time'] = 0
        
        return stats


Initialize sender

In [13]:
sender = JsonListApiSender(
    api_user=API_USER,
    api_base_url=API_BASE_URL,
    headers=HEADERS,
    retry_attempts=RETRY_ATTEMPTS,
    delay_between_requests=DELAY_BETWEEN_REQUESTS,
    verbose=VERBOSE,
    # Throttling configuration
    max_requests_per_second=MAX_REQUESTS_PER_SECOND,
    max_requests_per_minute=MAX_REQUESTS_PER_MINUTE,
    max_requests_per_hour=MAX_REQUESTS_PER_HOUR,
    respect_retry_after=RESPECT_RETRY_AFTER,
    adaptive_throttling=ADAPTIVE_THROTTLING
)

print("JsonListApiSender initialized with throttling configuration:")
if MAX_REQUESTS_PER_SECOND:
    print(f"  - Max {MAX_REQUESTS_PER_SECOND} requests/second")
if MAX_REQUESTS_PER_MINUTE:
    print(f"  - Max {MAX_REQUESTS_PER_MINUTE} requests/minute")
if MAX_REQUESTS_PER_HOUR:
    print(f"  - Max {MAX_REQUESTS_PER_HOUR} requests/hour")
if ADAPTIVE_THROTTLING:
    print(f"  - Adaptive throttling: enabled")
if RESPECT_RETRY_AFTER:
    print(f"  - Retry-After header support: enabled")

JsonListApiSender initialized with throttling configuration:
  - Max 5.0 requests/second
  - Max 250 requests/minute
  - Adaptive throttling: enabled
  - Retry-After header support: enabled


### 4.1 Build upload payloads for all providers

In [14]:
# Build LLM instance configurations from metadata
print("="*80)
print("Building LLM Instance Configurations")
print("="*80)
llm_instance_configs = build_instance_configs(
    embeddings_dict=embeddings_dict,
    metadata=metadata,
    provider_instance_mapping=PROVIDER_INSTANCE_MAPPING,
    instance_configs=INSTANCE_CONFIGS
)

# Build JSON objects for all providers
print("\n" + "="*80)
print("Building Upload Payloads")
print("="*80)
all_json_objects = build_all_json_objects(
    docs_df=docs,
    embeddings_dict=embeddings_dict,
    provider_project_mapping=PROVIDER_PROJECT_MAPPING,
    provider_instance_mapping=PROVIDER_INSTANCE_MAPPING,
    instance_owner=INSTANCE_OWNER,
    api_user=API_USER,
    max_objects=max_documents
)

print("\n" + "="*80)
print("JSON Objects Summary")
print("="*80)
for provider, objects in all_json_objects.items():
    print(f"{provider}: {len(objects)} objects ready for upload")

Building LLM Instance Configurations
LLM instance config for openai_text-embedding-3-small:
  Handle: openai-small, Model: text-embedding-3-small, Dimensions: 1536, API: openai
  Definition: _system/openai-small
LLM instance config for cohere_embed-v4.0_clustering:
  Handle: cohere-v4, Model: embed-v4.0, Dimensions: 1536, API: cohere
  Definition: _system/cohere-v4
LLM instance config for google_gemini-embedding-001_SEMANTIC_SIMILARITY:
  Handle: gemini-001, Model: gemini-embedding-001, Dimensions: 1536, API: gemini
  Definition: _system/gemini-embedding-001

Building Upload Payloads


Building payloads for openai_text-embedding-3-small: 100%|██████████| 20/20 [00:00<00:00, 24244.53it/s]


Built 20 JSON objects for openai_text-embedding-3-small


Building payloads for cohere_embed-v4.0_clustering: 100%|██████████| 20/20 [00:00<00:00, 25312.64it/s]


Built 20 JSON objects for cohere_embed-v4.0_clustering


Building payloads for google_gemini-embedding-001_SEMANTIC_SIMILARITY: 100%|██████████| 20/20 [00:00<00:00, 31823.25it/s]

Built 20 JSON objects for google_gemini-embedding-001_SEMANTIC_SIMILARITY

JSON Objects Summary
openai_text-embedding-3-small: 20 objects ready for upload
cohere_embed-v4.0_clustering: 20 objects ready for upload
google_gemini-embedding-001_SEMANTIC_SIMILARITY: 20 objects ready for upload


### 4.2 Upload to Vector Database

#### 4.1.1 Capture Initial EmbAPI State

Query the EmbAPI database for projects and LLM instances before uploading, so we can compare the state later.

In [15]:
# Query EmbAPI for current state before uploading
print("="*80)
print("QUERYING EMBAPI FOR INITIAL STATE")
print("="*80)

initial_projects = {}
initial_llm_instances = {}

# Query each project individually to get detailed information including embedding counts
print(f"\nQuerying individual projects from PROVIDER_PROJECT_MAPPING...")
for provider_name, project_handle in PROVIDER_PROJECT_MAPPING.items():
    project_info = sender.get_project_info(project_handle)
    if project_info:
        initial_projects[project_handle] = project_info
        print(f"  - {project_handle}: {project_info.get('num_embeddings', 0)} embeddings")
    else:
        print(f"  - {project_handle}: Project not found or error retrieving info")

# Get LLM instances
initial_llm_instances_response = sender.get_all_llm_instances()
if initial_llm_instances_response:
    instances = initial_llm_instances_response.get('llm_instances', [])
    initial_llm_instances = {i['instance_handle']: i for i in instances}
    print(f"\nCurrent LLM instances in EmbAPI: {len(instances)}")
    for inst in instances:
        print(f"  - {inst.get('instance_handle')}: {inst.get('model')}")
else:
    print("\nCould not fetch initial LLM instance state")

QUERYING EMBAPI FOR INITIAL STATE

Current projects in EmbAPI: 5
  - sal-openai-large: 0 embeddings
  - sal-openai-large: 0 embeddings
  - sal-openai-small: 0 embeddings
  - sal-cohere: 0 embeddings
  - sal-gemini-simil: 0 embeddings

Current LLM instances in EmbAPI: 0


In [16]:
# Upload all providers to their respective EmbAPI projects
# Use max_documents above in section "1.3.4 Limits" to limit
# the number of documents to upload per provider.
# Set max_documents to -1 to upload all documents
upload_results = sender.send_all_providers(
    provider_json_objects=all_json_objects,
    provider_project_mapping=PROVIDER_PROJECT_MAPPING,
    provider_instance_mapping=PROVIDER_INSTANCE_MAPPING,
    instance_owner=INSTANCE_OWNER,
    instance_configs=llm_instance_configs,
    num_items=max_documents if max_documents > 0 else None
)

Starting multi-provider upload
openai_text-embedding-3-small -> openai-small
  Items to upload: 20
Creating project 'openai-small'...


2026-02-10 15:33:10,217 - __main__ - INFO - Starting to send 20 items to https://c100-188.cloud.gwdg.de/embapi/v1/embeddings/sal/openai-small


✓ Created project 'openai-small' (ID: 12)
  Description: Embeddings from Openai provider with text-embedding-3-small model
  Instance: sal/openai-small

Uploading 20 items to project 'openai-small'...
Rate limits: 5.0/sec, 250/min


Uploading to openai-small: 100%|██████████| 20/20 [00:30<00:00,  1.53s/it]
2026-02-10 15:33:40,798 - __main__ - INFO - Upload complete for openai-small. Sent: 20, Successful: 20, Failed: 0


  ✓ Successful: 20
cohere_embed-v4.0_clustering -> cohere
  Items to upload: 20
Creating project 'cohere'...
✓ Created project 'cohere' (ID: 13)
  Description: Embeddings from Cohere provider with embed-v4.0_clustering model
  Instance: sal/cohere-v4
Rate limits: 5.0/sec, 250/min


Uploading to cohere: 100%|██████████| 20/20 [00:08<00:00,  2.30it/s]
2026-02-10 15:33:49,788 - __main__ - INFO - Upload complete for cohere. Sent: 20, Successful: 20, Failed: 0


  ✓ Successful: 20
google_gemini-embedding-001_SEMANTIC_SIMILARITY -> gemini-simil
  Items to upload: 20
Creating project 'gemini-simil'...
✓ Created project 'gemini-simil' (ID: 14)
  Description: Embeddings from Google provider with gemini-embedding-001_SEMANTIC_SIMILARITY model
  Instance: sal/gemini-001
Rate limits: 5.0/sec, 250/min


Uploading to gemini-simil: 100%|██████████| 20/20 [00:08<00:00,  2.23it/s]
2026-02-10 15:33:59,031 - __main__ - INFO - Upload complete for gemini-simil. Sent: 20, Successful: 20, Failed: 0


  ✓ Successful: 20
Upload Summary
openai_text-embedding-3-small:
  Total: 20, Success: 20, Failed: 0
cohere_embed-v4.0_clustering:
  Total: 20, Success: 20, Failed: 0
google_gemini-embedding-001_SEMANTIC_SIMILARITY:
  Total: 20, Success: 20, Failed: 0


### 4.3 Optional: Upload specific providers only

If you want to upload only specific providers, you can filter the embeddings and objects before uploading.

In [ ]:
# Example: Upload only specific providers
# Uncomment and modify this section to upload only selected providers

# selected_providers = ["openai_text-embedding-3-small", "cohere_embed-v4.0_clustering"]
# selected_json_objects = {k: v for k, v in all_json_objects.items() if k in selected_providers}
# selected_instance_configs = {k: v for k, v in llm_instance_configs.items() if k in selected_providers}

# upload_results = sender.send_all_providers(
#     provider_json_objects=selected_json_objects,
#     provider_project_mapping=PROVIDER_PROJECT_MAPPING,
#     provider_instance_mapping=PROVIDER_INSTANCE_MAPPING,
#     instance_owner=INSTANCE_OWNER,
#     instance_configs=selected_instance_configs,
#     num_items=max_documents if max_documents > 0 else None
# )

## 5. Upload Diagnostics and Reporting

Detailed analysis of the upload process including timing statistics, error analysis, and EmbAPI project status.

### 5.1 Upload Statistics by Provider

In [17]:
# Display detailed upload statistics by provider
print("="*80)
print("UPLOAD STATISTICS BY PROVIDER")
print("="*80)

for provider_name, result in upload_results.items():
    print(f"\n{provider_name}")
    print("-" * 60)
    print(f"  Project:               {result.get('project', 'N/A')}")
    print(f"  Total attempts:        {result.get('total_sent', 0)}")
    print(f"  Successful uploads:    {result.get('successful', 0)}")
    print(f"  Failed uploads:        {result.get('failed', 0)}")
    
    if result.get('failed', 0) > 0:
        print(f"  Success rate:          {result.get('successful', 0) / result.get('total_sent', 1) * 100:.1f}%")
        if result.get('failed_ids'):
            print(f"  Failed IDs:            {', '.join(str(x) for x in result['failed_ids'][:5])}")
            if len(result['failed_ids']) > 5:
                print(f"                         ... and {len(result['failed_ids']) - 5} more")
    else:
        print(f"  Success rate:          100.0%")
    
    if 'upload_duration_seconds' in result:
        duration = result['upload_duration_seconds']
        print(f"  Upload duration:       {duration:.2f}s ({duration/60:.1f} min)")
        if result.get('total_sent', 0) > 0:
            print(f"  Avg time per upload:   {duration / result['total_sent']:.3f}s")
    
    if 'error' in result:
        print(f"  Error:                 {result['error']}")

print("\n" + "="*80)

UPLOAD STATISTICS BY PROVIDER

openai_text-embedding-3-small
------------------------------------------------------------
  Project:               openai-small
  Total attempts:        20
  Successful uploads:    20
  Failed uploads:        0
  Success rate:          100.0%
  Upload duration:       30.58s (0.5 min)
  Avg time per upload:   1.529s

cohere_embed-v4.0_clustering
------------------------------------------------------------
  Project:               cohere
  Total attempts:        20
  Successful uploads:    20
  Failed uploads:        0
  Success rate:          100.0%
  Upload duration:       8.71s (0.1 min)
  Avg time per upload:   0.435s

google_gemini-embedding-001_SEMANTIC_SIMILARITY
------------------------------------------------------------
  Project:               gemini-simil
  Total attempts:        20
  Successful uploads:    20
  Failed uploads:        0
  Success rate:          100.0%
  Upload duration:       8.96s (0.1 min)
  Avg time per upload:   0.448s



### 5.2 Overall Upload Statistics and Error Analysis

In [18]:
# Get comprehensive upload statistics from sender
stats = sender.get_upload_statistics()

print("="*80)
print("OVERALL UPLOAD STATISTICS")
print("="*80)

print(f"\nTotal Requests:        {stats['total_requests']}")
print(f"Successful:            {stats['successful_requests']}")
print(f"Failed:                {stats['failed_requests']}")

if stats['total_requests'] > 0:
    success_rate = (stats['successful_requests'] / stats['total_requests']) * 100
    print(f"Success Rate:          {success_rate:.2f}%")

print(f"\n{'TIMING STATISTICS'}")
print("-" * 60)
print(f"Total Elapsed Time:    {stats['total_elapsed_time']:.2f}s ({stats['total_elapsed_time']/60:.1f} min)")
print(f"Total Request Time:    {stats['total_request_time']:.2f}s")
print(f"Average Request Time:  {stats['avg_request_time']:.3f}s")
print(f"Max Request Time:      {stats['max_request_time']:.3f}s")
print(f"Min Request Time:      {stats['min_request_time']:.3f}s")

if stats['total_elapsed_time'] > 0:
    throughput = stats['total_requests'] / stats['total_elapsed_time']
    print(f"Throughput:            {throughput:.2f} requests/second")

# Error analysis
if stats['error_codes']:
    print(f"\n{'ERROR CODE ANALYSIS'}")
    print("-" * 60)
    for error_code, count in sorted(stats['error_codes'].items()):
        print(f"  HTTP {error_code}:           {count} occurrences")
else:
    print(f"\n{'ERROR CODE ANALYSIS'}")
    print("-" * 60)
    print("  No errors encountered ✓")

print("\n" + "="*80)

OVERALL UPLOAD STATISTICS

Total Requests:        60
Successful:            60
Failed:                0
Success Rate:          100.00%

TIMING STATISTICS
------------------------------------------------------------
Total Elapsed Time:    48.81s (0.8 min)
Total Request Time:    48.10s
Average Request Time:  0.802s
Max Request Time:      7.484s
Min Request Time:      0.300s
Throughput:            1.23 requests/second

ERROR CODE ANALYSIS
------------------------------------------------------------
  No errors encountered ✓



### 5.3 EmbAPI Project Status

Query the EmbAPI database to get current status of projects that received embeddings in this upload.

In [19]:
# Query EmbAPI for detailed project information
uploaded_project_handles = {result['project'] for result in upload_results.values() if 'project' in result}

print("="*80)
print("EMBAPI PROJECT STATUS (Projects from this upload)")
print("="*80)

if uploaded_project_handles:
    relevant_projects = []
    for project_handle in uploaded_project_handles:
        project_info = sender.get_project_info(project_handle)
        if project_info:
            relevant_projects.append(project_info)
    
    if relevant_projects:
        for project in sorted(relevant_projects, key=lambda x: x['project_handle']):
            print(f"\n{project['project_handle']}")
            print("-" * 60)
            print(f"  Project ID:            {project.get('project_id', 'N/A')}")
            print(f"  Owner:                 {project.get('owner', 'N/A')}")
            print(f"  Description:           {project.get('description', 'N/A')}")
            print(f"  Number of embeddings:  {project.get('num_embeddings', 0)}")
            
            # Show connected LLM service instance
            instance_owner = project.get('instance_owner', 'N/A')
            instance_handle = project.get('instance_handle', 'N/A')
            print(f"  LLM Service Instance:  {instance_owner}/{instance_handle}")
            
            # Show authorized readers
            readers = project.get('authorizedReaders', [])
            if readers:
                if '*' in readers:
                    print(f"  Authorized readers:    All users (*)")
                else:
                    print(f"  Authorized readers:    {', '.join(readers)}")
            else:
                print(f"  Authorized readers:    None")
        
        print("\n" + "="*80)
        
        # Summary statistics
        total_embeddings = sum(p.get('num_embeddings', 0) for p in relevant_projects)
        print(f"\nSummary:")
        print(f"  Total projects:        {len(relevant_projects)}")
        print(f"  Total embeddings:      {total_embeddings}")
        print("="*80)
    else:
        print("\nFailed to retrieve information for any projects.")
        print("="*80)
else:
    print("\nNo projects uploaded in this session.")
    print("="*80)

EMBAPI PROJECT STATUS (Projects from this upload)

cohere
------------------------------------------------------------
  Project ID:            13
  Owner:                 sal
  Description:           N/A
  Number of embeddings:  0
  Authorized readers:    None

gemini-simil
------------------------------------------------------------
  Project ID:            14
  Owner:                 sal
  Description:           N/A
  Number of embeddings:  0
  Authorized readers:    None

openai-small
------------------------------------------------------------
  Project ID:            12
  Owner:                 sal
  Description:           N/A
  Number of embeddings:  0
  Authorized readers:    None


Summary:
  Total projects:        3
  Total embeddings:      0


### 5.4 Complete EmbAPI Response

Full JSON response from the EmbAPI projects API for reference.

In [20]:
# Display the complete JSON response from EmbAPI
if all_projects_response:

    # Filter to show only relevant projects in the full response
    filtered_response = {
        "$schema": all_projects_response.get("$schema", ""),
        "projects": relevant_projects
    }
    
    print("="*80)
    print("COMPLETE EMBAPI RESPONSE (Filtered to uploaded projects)")
    print("="*80)
    print(json.dumps(filtered_response, indent=2))
    print("="*80)
else:
    print("No EmbAPI response available to display.")

COMPLETE EMBAPI RESPONSE (Filtered to uploaded projects)
{
  "$schema": "https://c100-188.cloud.gwdg.de/schemas/GetProjectsResponseBody.json",
  "projects": [
    {
      "owner": "sal",
      "project_handle": "openai-small",
      "project_id": 12,
      "public_read": false,
      "role": "owner"
    },
    {
      "owner": "sal",
      "project_handle": "cohere",
      "project_id": 13,
      "public_read": false,
      "role": "owner"
    },
    {
      "owner": "sal",
      "project_handle": "gemini-simil",
      "project_id": 14,
      "public_read": false,
      "role": "owner"
    }
  ]
}


## 6. Changes Summary: Before vs After

Compare the initial EmbAPI state (before upload) with the final state to show what was created or modified.

### 6.1 Query Final EmbAPI State

In [21]:
# Query EmbAPI for final state after uploading
print("="*80)
print("QUERYING EMBAPI FOR FINAL STATE")
print("="*80)

final_projects = {}
final_llm_instances = {}

# Query each project individually to get detailed information including embedding counts
print(f"\nQuerying individual projects from PROVIDER_PROJECT_MAPPING...")
for provider_name, project_handle in PROVIDER_PROJECT_MAPPING.items():
    project_info = sender.get_project_info(project_handle)
    if project_info:
        final_projects[project_handle] = project_info
        print(f"  - {project_handle}: {project_info.get('num_embeddings', 0)} embeddings")
    else:
        print(f"  - {project_handle}: Project not found or error retrieving info")

# Get LLM instances
final_llm_instances_response = sender.get_all_llm_instances()
if final_llm_instances_response:
    instances = final_llm_instances_response.get('llm_instances', [])
    final_llm_instances = {i['instance_handle']: i for i in instances}
    print(f"\nCurrent LLM instances in EmbAPI: {len(instances)}")
    for inst in instances:
        print(f"  - {inst.get('instance_handle')}: {inst.get('model')}")
else:
    final_llm_instances = {}
    print("\nCould not fetch final LLM instance state")

QUERYING EMBAPI FOR FINAL STATE

Current projects in EmbAPI: 8
  - sal-openai-large: 0 embeddings
  - sal-openai-large: 0 embeddings
  - sal-openai-small: 0 embeddings
  - sal-cohere: 0 embeddings
  - sal-gemini-simil: 0 embeddings
  - openai-small: 0 embeddings
  - cohere: 0 embeddings
  - gemini-simil: 0 embeddings

Current LLM instances in EmbAPI: 0


### 6.2 Compare and Report Changes

In [22]:
# Compare initial and final states
print("="*80)
print("CHANGES SUMMARY: INITIAL vs FINAL STATE")
print("="*80)

# Analyze Projects
print("\n" + "="*80)
print("PROJECTS ANALYSIS")
print("="*80)

new_projects = set(final_projects.keys()) - set(initial_projects.keys())
modified_projects = set(final_projects.keys()) & set(initial_projects.keys())
removed_projects = set(initial_projects.keys()) - set(final_projects.keys())

if new_projects:
    print(f"\n✓ NEWLY CREATED PROJECTS ({len(new_projects)}):")
    for handle in sorted(new_projects):
        proj = final_projects[handle]
        print(f"  • {handle}")
        print(f"    ID: {proj['project_id']}, Embeddings: {proj['num_embeddings']}")
        print(f"    Description: {proj['description']}")
else:
    print("\n✗ No new projects created")

if modified_projects:
    print(f"\n📊 MODIFIED PROJECTS (embedding count changes):")
    embeddings_added_total = 0
    for handle in sorted(modified_projects):
        initial = initial_projects[handle]
        final = final_projects[handle]
        initial_count = initial['num_embeddings']
        final_count = final['num_embeddings']
        
        if final_count != initial_count:
            delta = final_count - initial_count
            embeddings_added_total += delta
            print(f"  • {handle}")
            print(f"    Before: {initial_count} embeddings")
            print(f"    After:  {final_count} embeddings")
            print(f"    Change: {delta:+d} embeddings")
    
    if embeddings_added_total == 0:
        print("  (No embedding count changes detected)")
    else:
        print(f"\n  Total embeddings added across all projects: {embeddings_added_total:+d}")
else:
    print("\n(No existing projects in this upload)")

if removed_projects:
    print(f"\n⚠ REMOVED PROJECTS ({len(removed_projects)}):")
    for handle in sorted(removed_projects):
        print(f"  • {handle}")
else:
    print("\n✓ No projects removed")

# Analyze LLM Instances
print("\n" + "="*80)
print("LLM INSTANCES ANALYSIS")
print("="*80)

new_llm_instances = set(final_llm_instances.keys()) - set(initial_llm_instances.keys())
modified_llm_instances = set(final_llm_instances.keys()) & set(initial_llm_instances.keys())
removed_llm_instances = set(initial_llm_instances.keys()) - set(final_llm_instances.keys())

if new_llm_instances:
    print(f"\n✓ NEWLY CREATED LLM INSTANCES ({len(new_llm_instances)}):")
    for handle in sorted(new_llm_instances):
        svc = final_llm_instances[handle]
        print(f"  • {handle}")
        print(f"    ID: {svc['llm_service_id']}, Model: {svc['model']}, Dimensions: {svc['dimensions']}")
else:
    print("\n✗ No new LLM instances created")

if modified_llm_instances:
    print(f"\n📊 EXISTING LLM INSTANCES (unchanged):")
    for handle in sorted(modified_llm_instances):
        svc = final_llm_instances[handle]
        print(f"  • {handle} (ID: {svc['llm_service_id']}, Model: {svc['model']})")
else:
    print("\n(No existing LLM instances)")

if removed_llm_instances:
    print(f"\n⚠ REMOVED LLM INSTANCES ({len(removed_llm_instances)}):")
    for handle in sorted(removed_llm_instances):
        print(f"  • {handle}")
else:
    print("\n✓ No LLM instances removed")

# Overall Summary
print("\n" + "="*80)
print("OVERALL SUMMARY")
print("="*80)
print(f"\nProjects:")
print(f"  Created: {len(new_projects)}")
print(f"  Modified: {len([h for h in modified_projects if final_projects[h]['num_embeddings'] != initial_projects[h]['num_embeddings']])}")
print(f"  Removed: {len(removed_projects)}")
print(f"  Total in EmbAPI: {len(final_projects)}")

print(f"\nLLM Instances:")
print(f"  Created: {len(new_llm_instances)}")
print(f"  Existing: {len(modified_llm_instances)}")
print(f"  Removed: {len(removed_llm_instances)}")
print(f"  Total in EmbAPI: {len(final_llm_instances)}")

print(f"\nEmbeddings:")
initial_total = sum(p['num_embeddings'] for p in initial_projects.values())
final_total = sum(p['num_embeddings'] for p in final_projects.values())
print(f"  Before upload: {initial_total}")
print(f"  After upload:  {final_total}")
print(f"  Net change:    {final_total - initial_total:+d}")

print("\n" + "="*80)

CHANGES SUMMARY: INITIAL vs FINAL STATE

PROJECTS ANALYSIS

✓ NEWLY CREATED PROJECTS (3):
  • cohere


KeyError: 'number_of_embeddings'